# **Configuration and Imports**

## Environment & Imports

In [1]:
import os
import gc
import sys
import glob
import time
import json
import math
import random
import warnings
from abc import ABCMeta, abstractmethod

import numpy as np
import pandas as pd
import networkx as nx
from scipy import sparse

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import RDLogger

# Suppress RDKit noisy parse warnings
RDLogger.DisableLog('rdApp.*')
warnings.filterwarnings('ignore', category=UserWarning)

import mxnet as mx
from mxnet import gluon, autograd, nd
from mxnet.gluon import nn
from mxnet.gluon.data import Dataset, DataLoader
from mxnet.gluon.data.sampler import Sampler
from mxnet.autograd import Function

/home/muhamed/.conda/envs/ctddg_env/lib/python3.9/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


## Paths, Execution Toggles, and Hyperparameters

In [2]:
FORCE_RESTART = True

GPU_ID = 0
CTX = mx.gpu(GPU_ID) if mx.context.num_gpus() > 0 else mx.cpu()
print(f"Active Compute Context: {CTX}")

Active Compute Context: gpu(0)


In [3]:
DATA_PATH = "/home/muhamed/repo/CTDDG/datasets/data/chembl/chembl_toxicity_labels.csv"
ATOM_TYPES_PATH = "/home/muhamed/repo/CTDDG/datasets/data/atom_types.txt"
CHECKPOINT_DIR = "/home/muhamed/repo/CTDDG/checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Standard checkpoint artifact filenames
CKPT_PARAMS_FILE = os.path.join(CHECKPOINT_DIR, "ckpt.params")
CKPT_TRAINER_FILE = os.path.join(CHECKPOINT_DIR, "trainer.status")
CKPT_CONFIG_FILE = os.path.join(CHECKPOINT_DIR, "configs.json")
LOG_FILE = os.path.join(CHECKPOINT_DIR, "log.out")



In [4]:
# ==============================================================================
# 3. Model Architecture & Training Hyperparameters
# ==============================================================================
# Tunable scaling parameters for L40 GPU (48GB VRAM)
BATCH_SIZE = 16          # Original paper used 8. Can scale to 16, 32, or 64 on L40
K_PATHS = 5              # K in CTDDG paper (stochastic generation paths per molecule)
P_ALPHA = 0.8            # Randomness parameter alpha in proposal Q_alpha (paper = 0.8)
LAMBDA_TOX = 1.0         # Toxicity penalty parameter lambda in CTDDG Eq. 8

# Graph Neural Network Dimensions
F_E = 16                 # Atom embedding size
F_H = [32, 64, 128, 128, 256, 256]  # Output sizes of the 6 GCN layers
F_SKIP = 256             # Output size of skip-connection layer
F_C = [512]              # Hidden sizes of post-GCN fully connected layers
FH_POLICY = 128          # Hidden size for the policy head
ACTIVATION = 'relu'      # GCN activation function
N_RNN = 3                # Number of GRU layers
D_RECEPTIVE = 2          # Graph receptive distance (adds D_2 and D_3 neighborhoods)
MAX_ITER = 120           # Cap on generation/action steps per molecule

# Optimizer & Scheduling
LR = 1e-3                # Initial learning rate
DECAY = 1e-3             # Weight decay rate
DECAY_STEP = 100         # Step interval for learning rate decay
CLIP_GRAD = 3.0          # Gradient clipping threshold
SUMMARY_STEP = 5       # Step interval to log loss and save checkpoints
TOTAL_ITERATIONS = 480000# Total training steps

## Verification & Sanity Check

In [5]:
# Verify path accessibility and GPU tensor allocation
assert os.path.exists(DATA_PATH), f"Data file not found at: {DATA_PATH}"
assert os.path.exists(ATOM_TYPES_PATH), f"Atom types file not found at: {ATOM_TYPES_PATH}"

# Test GPU tensor allocation
test_gpu_tensor = nd.zeros((1,), ctx=CTX)
test_gpu_tensor.asnumpy()

print(f"[OK] Paths verified.")
print(f"[OK] MXNet GPU context {CTX} is fully responsive.")
print(f"FORCE_RESTART is set to: {FORCE_RESTART}")

[OK] Paths verified.
[OK] MXNet GPU context gpu(0) is fully responsive.
FORCE_RESTART is set to: True


# **Data Loading & Batching**

## Molecular Specification & Graph Traversal Utilities

In [6]:
class MoleculeSpec(object):
    """Encodes and decodes atom/bond types using the immutable atom vocabulary."""
    def __init__(self, file_name=ATOM_TYPES_PATH):
        self.atom_types = []
        self.atom_symbols = []
        with open(file_name, 'r') as f:
            for line in f:
                atom_type_i = line.strip('\n').split(',')
                self.atom_types.append((atom_type_i[0], int(atom_type_i[1]), int(atom_type_i[2])))
                if atom_type_i[0] not in self.atom_symbols:
                    self.atom_symbols.append(atom_type_i[0])
                    
        self.bond_orders = [
            Chem.BondType.AROMATIC,
            Chem.BondType.SINGLE,
            Chem.BondType.DOUBLE,
            Chem.BondType.TRIPLE
        ]
        self.max_iter = MAX_ITER

    def get_atom_type(self, atom):
        return self.atom_types.index((atom.GetSymbol(), atom.GetFormalCharge(), atom.GetNumExplicitHs()))

    def get_bond_type(self, bond):
        return self.bond_orders.index(bond.GetBondType())

    def index_to_atom(self, idx):
        symbol, charge, hs = self.atom_types[idx]
        a = Chem.Atom(symbol)
        a.SetFormalCharge(charge)
        a.SetNumExplicitHs(hs)
        return a

    def index_to_bond(self, mol, begin_id, end_id, idx):
        mol.AddBond(begin_id, end_id, self.bond_orders[idx])

    @property
    def num_atom_types(self):
        return len(self.atom_types)

    @property
    def num_bond_types(self):
        return len(self.bond_orders)

_mol_spec = None

def get_mol_spec():
    global _mol_spec
    if _mol_spec is None:
        _mol_spec = MoleculeSpec(ATOM_TYPES_PATH)
    return _mol_spec


def get_graph_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"Invalid SMILES string: {smiles}")

    atom_types, atom_ranks, bonds, bond_types = [], [], [], []
    for a, r in zip(mol.GetAtoms(), Chem.CanonicalRankAtoms(mol)):
        atom_types.append(get_mol_spec().get_atom_type(a))
        atom_ranks.append(r)
        
    for b in mol.GetBonds():
        idx_1, idx_2 = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        bt = get_mol_spec().get_bond_type(b)
        bonds.append([idx_1, idx_2])
        bond_types.append(bt)

    graph = nx.Graph()
    graph.add_nodes_from(range(len(atom_types)))
    graph.add_edges_from(bonds)
    return graph, atom_types, atom_ranks, bonds, bond_types


def traverse_graph(graph, atom_ranks, current_node=None, step_ids=None, p=P_ALPHA, log_p=0.0):
    """Importance-sampling proposal Q_alpha traversal using canonical rank."""
    if current_node is None:
        next_nodes = range(len(atom_ranks))
        step_ids = [-1] * len(next_nodes)
        next_node_ranks = atom_ranks
    else:
        next_nodes = list(graph.neighbors(current_node))
        next_nodes = [n for n in next_nodes if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]
        
    next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    while len(next_nodes) > 0:
        if len(next_nodes) == 1:
            next_node = next_nodes[0]
        elif random.random() >= (1.0 - p):
            next_node = next_nodes[0]
            log_p += np.log(p)
        else:
            next_node = next_nodes[random.randint(1, len(next_nodes) - 1)]
            log_p += np.log((1.0 - p) / (len(next_nodes) - 1))
            
        step_ids[next_node] = max(step_ids) + 1
        _, log_p = traverse_graph(graph, atom_ranks, next_node, step_ids, p, log_p)
        next_nodes = [n for n in graph.neighbors(next_node) if step_ids[n] < 0]
        next_node_ranks = [atom_ranks[n] for n in next_nodes]
        next_nodes = [n for n, r in sorted(zip(next_nodes, next_node_ranks), key=lambda x: x[1])]

    return step_ids, log_p


def single_reorder(X_0, A_0, step_ids):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    step_ids = np.array(step_ids, dtype=np.int32)
    sorted_ids = np.argsort(step_ids)
    X_0 = X_0[sorted_ids]
    A_0[:, 0], A_0[:, 1] = step_ids[A_0[:, 0]], step_ids[A_0[:, 1]]
    
    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    A_0 = A_0[np.lexsort([-min_b, max_b]), :]

    max_b, min_b = np.amax(A_0[:, :2], axis=1), np.amin(A_0[:, :2], axis=1)
    is_append = np.concatenate([np.array([True]), max_b[1:] > max_b[:-1]])
    A_0 = np.concatenate([
        np.where(is_append[:, np.newaxis], np.stack([min_b, max_b], axis=1), np.stack([max_b, min_b], axis=1)),
        A_0[:, -1:]
    ], axis=1)
    return X_0, A_0


def single_expand(X_0, A_0):
    X_0, A_0 = np.copy(X_0), np.copy(A_0)
    is_append_iter = np.less(A_0[:, 0], A_0[:, 1]).astype(np.int32)

    NX = np.cumsum(np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1))
    shift = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    X_index = np.arange(NX.sum(), dtype=np.int32) - np.repeat(shift, NX)
    X = X_0[X_index]

    _, A_index = np.tril_indices(A_0.shape[0])
    A = A_0[A_index, :]
    NA = np.arange(A_0.shape[0] + 1)

    action_type = 1 - is_append_iter
    atom_type = np.where(action_type == 0, X_0[A_0[:, 1]], 0)
    bond_type = A_0[:, 2]
    append_pos = np.where(action_type == 0, A_0[:, 0], 0)
    connect_pos = np.where(action_type == 1, A_0[:, 1], 0)
    actions = np.stack([action_type, atom_type, bond_type, append_pos, connect_pos], axis=1)
    actions = np.append(actions, [[2, 0, 0, 0, 0]], axis=0)
    action_0 = np.array([X_0[0]], dtype=np.int32)

    last_atom_index = shift + NX - 1
    last_atom_mask = np.zeros_like(X)
    last_atom_mask[last_atom_index] = np.where(
        np.pad(is_append_iter, [[1, 0]], mode='constant', constant_values=1) == 1,
        np.ones_like(last_atom_index),
        np.ones_like(last_atom_index) * 2
    )
    return action_0, X, NX, A, NA, actions, last_atom_mask


def get_d(A, X):
    def _to_sparse(_A, _X):
        return sparse.coo_matrix(
            (np.ones([_A.shape[0] * 2], dtype=np.int32),
             (np.concatenate([_A[:, 0], _A[:, 1]], axis=0),
              np.concatenate([_A[:, 1], _A[:, 0]], axis=0))),
            shape=[_X.shape[0]] * 2
        )

    A_sparse = _to_sparse(A, X)
    d2 = A_sparse * A_sparse
    d3 = d2 * A_sparse

    D_2 = np.stack(d2.nonzero(), axis=1)
    D_2 = D_2[D_2[:, 0] < D_2[:, 1], :]

    D_3 = np.stack(d3.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]

    D_3_sparse = _to_sparse(D_3, X)
    D_3_sparse = D_3_sparse - D_3_sparse.multiply(A_sparse)
    D_3 = np.stack(D_3_sparse.nonzero(), axis=1)
    D_3 = D_3[D_3[:, 0] < D_3[:, 1], :]
    return D_2, D_3


def merge_single_0(X_0, A_0, NX_0, NA_0):
    cumsum = np.cumsum(np.pad(NX_0, [[1, 0]], mode='constant')[:-1])
    A_0[:, :2] += np.stack([np.repeat(cumsum, NA_0)] * 2, axis=1)
    D_0_2, D_0_3 = get_d(A_0, X_0)

    A_split = []
    for i in range(get_mol_spec().num_bond_types):
        A_split.append(A_0[A_0[:, 2] == i, :2])
    A_split.extend([D_0_2, D_0_3])
    A_0 = A_split
    NX_rep_0 = np.repeat(np.arange(NX_0.shape[0]), NX_0)
    return X_0, A_0, NX_0, NX_rep_0


def merge_single(X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p):
    X, A, NX, NX_rep = merge_single_0(X, A, NX, NA)
    cumsum = np.cumsum(np.pad(NX, [[1, 0]], mode='constant')[:-1])
    actions[:, -2] += cumsum * (actions[:, 0] == 0)
    actions[:, -1] += cumsum * (actions[:, 0] == 1)
    mol_ids_rep = np.repeat(mol_ids, NX)
    rep_ids_rep = np.repeat(rep_ids, NX)
    return (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p)

## Data Ingestion & Sampler

In [7]:
def load_chembl_dataset(file_path):
    """Loads SMILES and binary Ames labels from CSV or space-separated files."""
    try:
        df = pd.read_csv(file_path)
        # Auto-detect SMILES column
        smiles_col = next((c for c in df.columns if 'smiles' in c.lower()), df.columns[0])
        # Auto-detect Toxicity label column
        tox_col = next((c for c in df.columns if any(k in c.lower() for k in ['tox', 'label', 'ames', 'class'])), df.columns[1])
        
        smiles_list = df[smiles_col].astype(str).tolist()
        tox_list = df[tox_col].astype(int).tolist()
    except Exception:
        # Fallback for plain space/comma delimited files without standard headers
        smiles_list, tox_list = [], []
        with open(file_path, 'r') as f:
            for line in f:
                parts = line.strip().replace(',', ' ').split()
                if len(parts) >= 2:
                    smiles_list.append(parts[0])
                    tox_list.append(int(parts[1]))
                elif len(parts) == 1:
                    smiles_list.append(parts[0])
                    tox_list.append(0)

    records = list(zip(smiles_list, tox_list))
    return records


class BalancedSampler(Sampler):
    """Balances mini-batch lengths to minimize padding overhead during GCN/RNN computation."""
    def __init__(self, cost, batch_size):
        index = np.argsort(cost).tolist()
        chunk_size = int(float(len(cost)) / batch_size)
        self.index = []
        for i in range(batch_size):
            self.index.append(index[i * chunk_size:(i + 1) * chunk_size])

    def _g(self):
        for index_i in self.index:
            random.shuffle(index_i)
        for batch_index in zip(*self.index):
            yield list(batch_index)

    def __iter__(self):
        return self._g()

    def __len__(self):
        return len(self.index[0])

## Graph & RNN Data Loaders (With OOV Shield)

In [8]:
def process_single(record, k=K_PATHS, p=P_ALPHA):
    """Extracts graph, computes K proposal paths, and binds true Ames labels."""
    if isinstance(record, (tuple, list)):
        smiles, tox_label = record[0], int(record[1])
    elif " " in record:
        parts = record.split(" ")
        smiles, tox_label = parts[0], int(parts[1])
    else:
        smiles, tox_label = record, 0

    graph, atom_types, atom_ranks, bonds, bond_types = get_graph_from_smiles(smiles)
    X_0 = np.array(atom_types, dtype=np.int32)
    A_0 = np.concatenate([
        np.array(bonds, dtype=np.int32),
        np.array(bond_types, dtype=np.int32)[:, np.newaxis]
    ], axis=1) if len(bonds) > 0 else np.zeros((0, 3), dtype=np.int32)

    X, A, NX, NA = [], [], [], []
    mol_ids, rep_ids, iw_ids = [], [], []
    action_0, actions, last_append_mask = [], [], []
    log_p = []

    tox_class = np.full((k,), tox_label, dtype=np.int32)

    for i in range(k):
        step_ids_i, log_p_i = traverse_graph(graph, atom_ranks, p=p)
        X_i, A_i = single_reorder(X_0, A_0, step_ids_i)
        action_0_i, X_i, NX_i, A_i, NA_i, actions_i, last_atom_mask_i = single_expand(X_i, A_i)

        X.append(X_i)
        A.append(A_i)
        NX.append(NX_i)
        NA.append(NA_i)
        action_0.append(action_0_i)
        actions.append(actions_i)
        last_append_mask.append(last_atom_mask_i)
        mol_ids.append(np.zeros_like(NX_i, dtype=np.int32))
        rep_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        iw_ids.append(np.ones_like(NX_i, dtype=np.int32) * i)
        log_p.append(log_p_i)

    return (
        np.concatenate(X, axis=0), np.concatenate(A, axis=0),
        np.concatenate(NX, axis=0), np.concatenate(NA, axis=0),
        np.concatenate(mol_ids, axis=0), np.concatenate(rep_ids, axis=0),
        np.concatenate(iw_ids, axis=0), np.concatenate(action_0, axis=0),
        np.concatenate(actions, axis=0), np.concatenate(last_append_mask, axis=0),
        np.array(log_p, dtype=np.float32), tox_class
    )


class MolRNNCollate:
    """Standalone collator to avoid pickling the DataLoader's multiprocessing pool."""
    def __init__(self, k=K_PATHS, p=P_ALPHA):
        self.k = k
        self.p = p

    def __call__(self, batch):
        # 1. SHIELD: Pre-filter out any molecules with Out-Of-Vocabulary atoms
        valid_batch = []
        for record in batch:
            smiles = record[0] if isinstance(record, (tuple, list)) else (record.split()[0] if " " in record else record)
            try:
                _ = get_graph_from_smiles(smiles)
                valid_batch.append(record)
            except ValueError:
                continue # Safely skip OOV atoms
                
        if not valid_batch:
            return None 
            
        # 2. Process only the valid, guaranteed-to-parse records
        shapes = [[0], [0, 3], [0], [0], [0], [0], [0], [0], [0, 5], [0], [0], [0]]
        dtypes = [np.int32, np.int32, np.int32, np.int32, np.int32, np.int32, np.int32, np.int32, np.int32, np.int32, np.float32, np.int32]
        
        _build = lambda: [np.zeros(shape=s, dtype=d) for s, d in zip(shapes, dtypes)]
        _append = lambda _r0, _r1: [np.concatenate([__r0, __r1], axis=0) for __r0, __r1 in zip(_r0, _r1)]

        X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class = _build()

        valid_count = 0
        for record in valid_batch:
            (X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i,
             action_0_i, actions_i, last_append_mask_i, log_p_i, tox_class_i) = process_single(record, self.k, self.p)

            if valid_count != 0:
                mol_ids_i += mol_ids[-1] + 1
                iw_ids_i += iw_ids[-1] + 1

            (X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class) = _append(
                [X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p, tox_class],
                [X_i, A_i, NX_i, NA_i, mol_ids_i, rep_ids_i, iw_ids_i, action_0_i, actions_i, last_append_mask_i, log_p_i, tox_class_i.reshape(-1)]
            )
            valid_count += 1

        X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p = merge_single(
            X, A, NX, NA, mol_ids, rep_ids, iw_ids, action_0, actions, last_append_mask, log_p
        )
        result_out = [X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p, tox_class]
        
        # RNN mapping logic
        smiles_batch = [b[0] if isinstance(b, (tuple, list)) else b.split()[0] for b in valid_batch]
        mol_list = [Chem.MolFromSmiles(s) for s in smiles_batch]

        graph_to_rnn = np.zeros((len(valid_batch), self.k, get_mol_spec().max_iter), dtype=np.int32)
        rnn_to_graph = []
        cum_sum = 0
        
        for i, mol_i in enumerate(mol_list):
            num_iter = mol_i.GetNumBonds() + 1
            for k_idx in range(self.k):
                graph_to_rnn[i, k_idx, :num_iter] = np.arange(num_iter) + cum_sum
                rnn_to_graph_0 = np.ones([num_iter], dtype=np.int32) * i
                rnn_to_graph_1 = np.ones_like(rnn_to_graph_0) * k_idx
                rnn_to_graph_2 = np.arange(num_iter)
                rnn_to_graph.append(np.stack([rnn_to_graph_0, rnn_to_graph_1, rnn_to_graph_2], axis=0))
                cum_sum += num_iter

        rnn_to_graph = np.concatenate(rnn_to_graph, axis=1)
        NX_cum = np.cumsum(result_out[6])
        return result_out + [graph_to_rnn, rnn_to_graph, NX_cum]


def batch_to_tensor(record):
    """Standalone GPU tensor assignment function."""
    *base_records, graph_to_rnn, rnn_to_graph, NX_cum = record
    (X, A, mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask,
     NX, NX_rep, action_0, actions, log_p, tox_class) = base_records

    X = nd.array(X, ctx=CTX, dtype='int32')
    A_sparse = []
    for A_i in A:
        if A_i.shape[0] == 0:
            A_sparse.append(None)
        else:
            A_i = np.concatenate([A_i, A_i[:, [1, 0]]], axis=0)
            data = np.ones((A_i.shape[0],), dtype=np.float32)
            row, col = A_i[:, 0], A_i[:, 1]
            A_sparse_i = nd.sparse.csr_matrix(
                (data, (row, col)), shape=(int(X.shape[0]), int(X.shape[0])), ctx=CTX, dtype='float32'
            )
            A_sparse.append(A_sparse_i)

    batch_size = (mol_ids_rep.max() + 1).item()
    iw_size = (rep_ids_rep.max() + 1).item()

    mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions = [
        nd.array(_x, ctx=CTX, dtype='int32') for _x in [mol_ids_rep, rep_ids_rep, iw_ids, last_append_mask, NX, NX_rep, action_0, actions]
    ]
    log_p, tox_class = nd.array(log_p, ctx=CTX, dtype='float32'), nd.array(tox_class, ctx=CTX, dtype='float32')
    
    graph_to_rnn = nd.array(graph_to_rnn, ctx=CTX, dtype='int32')
    rnn_to_graph = nd.array(rnn_to_graph, ctx=CTX, dtype='int32')
    NX_cum = nd.array(NX_cum, ctx=CTX, dtype='int32')
    
    return [X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p, batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum]

## Part 2 Verification & Mini-Batch Smoke Test

In [9]:
# 1. Ingest dataset
dataset = load_chembl_dataset(DATA_PATH)
print(f"Total molecules loaded: {len(dataset)}")

# Verify toxicity class distribution
labels = [rec[1] for rec in dataset]
num_toxic = sum(labels)
num_nontoxic = len(labels) - num_toxic
print(f"Class distribution: Non-Toxic (0) = {num_nontoxic}, Toxic (1) = {num_toxic}")

# 2. Build sampler and standard MXNet data loader
sampler_train = BalancedSampler(cost=[len(rec[0]) for rec in dataset], batch_size=BATCH_SIZE)

# Explicitly use standard MXNet DataLoader with our detached class
collate_fn = MolRNNCollate(k=K_PATHS, p=P_ALPHA)
loader_train = DataLoader(dataset, batch_sampler=sampler_train, num_workers=8, batchify_fn=collate_fn)

# 3. Pull a single batch and transfer to GPU
batch_cpu = next(iter(loader_train))
batch_gpu = batch_to_tensor(batch_cpu) # Using the new standalone function

(X, A_sparse, iw_ids, last_append_mask, NX, NX_rep, action_0, actions,
 log_p, batch_size, iw_size, tox_class, graph_to_rnn, rnn_to_graph, NX_cum) = batch_gpu

print("\n--- Mini-Batch Shape & Sanity Check ---")
print(f"Batch size (N): {batch_size}, Paths per molecule (K): {iw_size}")
print(f"Node features X shape: {X.shape}, Context: {X.context}")
print("Verification passed!!!")

Total molecules loaded: 1207359
Class distribution: Non-Toxic (0) = 919739, Toxic (1) = 287620

--- Mini-Batch Shape & Sanity Check ---
Batch size (N): 16, Paths per molecule (K): 5
Node features X shape: (28694,), Context: gpu(0)
Verification passed!!!


# **Model Classes, Eq. 8 Loss, & Checkpoint Loading Logic.**

## MXNet Graph Math & Primitives

In [10]:
class GraphConvFn(Function):
    def __init__(self, A):
        self.A = A
        self.A_T = self.A # Assume symmetric
        super(GraphConvFn, self).__init__()

    def forward(self, X):
        if self.A is not None:
            if len(X.shape) > 2:
                X_resized = X.reshape((X.shape[0], -1))
                output = nd.sparse.dot(self.A, X_resized)
                output = output.reshape([-1] + [X.shape[i] for i in range(1, len(X.shape))])
            else:
                output = nd.sparse.dot(self.A, X)
            return output
        return nd.zeros_like(X)

    def backward(self, grad_output):
        if self.A is not None:
            if len(grad_output.shape) > 2:
                grad_output_resized = grad_output.reshape((grad_output.shape[0], -1))
                grad_input = nd.sparse.dot(self.A_T, grad_output_resized)
                grad_input = grad_input.reshape([-1] + [grad_output.shape[i] for i in range(1, len(grad_output.shape))])
            else:
                grad_input = nd.sparse.dot(self.A_T, grad_output)
            return grad_input
        return nd.zeros_like(grad_output)


class EfficientGraphConvFn(Function):
    """Saves memory by re-computing forward activations during backward pass."""
    def __init__(self, A_list):
        self.A_list = A_list
        super(EfficientGraphConvFn, self).__init__()

    def forward(self, X, W):
        X_list = [X]
        for A in self.A_list:
            X_list.append(nd.sparse.dot(A, X) if A is not None else nd.zeros_like(X))
        X_out = nd.concat(*X_list, dim=1)
        self.save_for_backward(X, W)
        return nd.dot(X_out, W)

    def backward(self, grad_output):
        X, W = self.saved_tensors
        X_list = [X]
        for A in self.A_list:
            X_list.append(nd.sparse.dot(A, X) if A is not None else nd.zeros_like(X))
        X_out = nd.concat(*X_list, dim=1)
        
        grad_W = nd.dot(X_out.T, grad_output)
        grad_X_out = nd.dot(grad_output, W.T)
        
        # BUG FIX: Explicitly setting axis=1 for the split
        grad_X_out_list = nd.split(grad_X_out, num_outputs=len(self.A_list) + 1, axis=1)

        grad_X = [grad_X_out_list[0]]
        for A, grad_out in zip(self.A_list, grad_X_out_list[1:]):
            grad_X.append(nd.sparse.dot(A, grad_out) if A is not None else nd.zeros_like(grad_out))
            
        return sum(grad_X), grad_W


class SegmentSumFn(GraphConvFn):
    def __init__(self, idx, num_seg):
        data = nd.ones(idx.shape[0], ctx=idx.context, dtype='float32')
        row, col = idx, nd.arange(idx.shape[0], ctx=idx.context, dtype='int32')
        shape = (num_seg, int(idx.shape[0]))
        sparse_mat = nd.sparse.csr_matrix((data, (row, col)), shape=shape, ctx=idx.context, dtype='float32')
        super(SegmentSumFn, self).__init__(sparse_mat)
        self.A_T = nd.sparse.csr_matrix((data, (col, row)), shape=(shape[1], shape[0]), ctx=idx.context, dtype='float32')


def squeeze(input_tensor, axis):
    new_shape = list(input_tensor.shape)
    del new_shape[axis]
    return input_tensor.reshape(new_shape)

def unsqueeze(input_tensor, axis):
    return nd.expand_dims(input_tensor, axis=axis)

def logsumexp(inputs, axis=None, keepdims=False):
    if axis is None:
        inputs = inputs.reshape([-1])
        axis = 0
    s = nd.max(inputs, axis=axis, keepdims=True)
    outputs = s + (inputs - s).exp().sum(axis=axis, keepdims=True).log()
    if not keepdims:
        outputs = nd.sum(outputs, axis=axis, keepdims=False)
    return outputs

def get_activation(name):
    return {'relu': nd.relu, 'tanh': nd.tanh}[name]

## Neural Network Layers

In [11]:
class Linear_BN(nn.Sequential):
    def __init__(self, F_in, F_out):
        super(Linear_BN, self).__init__()
        self.add(nn.Dense(F_out, in_units=F_in, use_bias=False))
        self.add(BatchNorm(in_channels=F_out))


class GraphConv(nn.Block):
    def __init__(self, Fin, Fout, D):
        super(GraphConv, self).__init__()
        self.Fin = Fin
        self.Fout = Fout
        self.D = D
        self.W = self.params.get('w', shape=(self.Fin * (self.D + 1), self.Fout), allow_deferred_init=False)

    def forward(self, X, A_list):
        return EfficientGraphConvFn(A_list)(X, self.W.data(X.context))


class Policy(nn.Block):
    def __init__(self, F_in, F_h, N_A, N_B, k=1):
        super(Policy, self).__init__()
        self.F_in, self.F_h, self.N_A, self.N_B, self.k = F_in, F_h, N_A, N_B, k
        with self.name_scope():
            self.linear_h = Linear_BN(F_in * 2, self.F_h * k)
            self.linear_h_t = Linear_BN(F_in, self.F_h * k)
            self.linear_x = nn.Dense(self.N_B + self.N_B * self.N_A, in_units=self.F_h)
            self.linear_x_t = nn.Dense(1, in_units=self.F_h)
            self.linear_pi = nn.Dense(self.k, in_units=self.F_in) if self.k > 1 else None

    def forward(self, X, NX, NX_rep, X_end=None):
        if X_end is None:
            X_end = SegmentSumFn(NX_rep, NX.shape[0])(X) / nd.cast(unsqueeze(NX, 1), 'float32')
        X = nd.concat(X, X_end[NX_rep, :], dim=1)

        X_h = nd.relu(self.linear_h(X)).reshape([-1, self.F_h])
        X_h_end = nd.relu(self.linear_h_t(X_end)).reshape([-1, self.F_h])

        # BUG FIX: Get raw logits and clip them between -80 and 80 to prevent float32 exp() overflow
        raw_logits_x = self.linear_x(X_h).reshape([-1, self.k, self.N_B + self.N_B * self.N_A])
        raw_logits_x_t = self.linear_x_t(X_h_end).reshape([-1, self.k, 1])
        
        safe_logits_x = nd.clip(raw_logits_x, -80.0, 80.0)
        safe_logits_x_t = nd.clip(raw_logits_x_t, -80.0, 80.0)

        # Proceed with exponentiation safely
        X_x = nd.exp(safe_logits_x)
        X_x_end = nd.exp(safe_logits_x_t)

        X_sum = nd.sum(SegmentSumFn(NX_rep, NX.shape[0])(X_x), -1, keepdims=True) + X_x_end
        X_sum_gathered = X_sum[NX_rep, :, :]

        X_softmax = X_x / (X_sum_gathered + 1e-10)
        X_softmax_end = X_x_end / (X_sum + 1e-10)

        if self.k > 1:
            pi = unsqueeze(nd.softmax(self.linear_pi(X_end), axis=1), -1)
            pi_gathered = pi[NX_rep, :, :]
            X_softmax = nd.sum(X_softmax * pi_gathered, axis=1)
            X_softmax_end = nd.sum(X_softmax_end * pi, axis=1)
        else:
            X_softmax = squeeze(X_softmax, 1)
            X_softmax_end = squeeze(X_softmax_end, 1)

        connect, append = X_softmax[:, :self.N_B], X_softmax[:, self.N_B:]
        append = append.reshape([-1, self.N_A, self.N_B])
        end = squeeze(X_softmax_end, -1)
        return append, connect, end


class BatchNorm(nn.Block):
    def __init__(self, in_channels, momentum=0.9, eps=1e-5):
        super(BatchNorm, self).__init__()
        self.F = in_channels
        self.bn_weight = self.params.get('bn_weight', shape=(self.F,), init=mx.init.One(), allow_deferred_init=False)
        self.bn_bias = self.params.get('bn_bias', shape=(self.F,), init=mx.init.Zero(), allow_deferred_init=False)
        self.running_mean = self.params.get('running_mean', grad_req='null', shape=(self.F,), init=mx.init.Zero(), allow_deferred_init=False, differentiable=False)
        self.running_var = self.params.get('running_var', grad_req='null', shape=(self.F,), init=mx.init.One(), allow_deferred_init=False, differentiable=False)
        self.momentum, self.eps = momentum, eps

    def forward(self, x):
        return nd.BatchNorm(x, gamma=self.bn_weight.data(x.context), beta=self.bn_bias.data(x.context),
                            moving_mean=self.running_mean.data(x.context), moving_var=self.running_var.data(x.context),
                            eps=self.eps, momentum=self.momentum, use_global_stats=not autograd.is_training())

## Generator & Toxicity Loss Fix

In [12]:
class MoleculeGenerator(nn.Block):
    __metaclass__ = ABCMeta

    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs):
        super(MoleculeGenerator, self).__init__()
        self.N_A, self.N_B, self.D = N_A, N_B, D
        self.F_e, self.F_skip, self.Fh_policy = F_e, F_skip, Fh_policy
        self.F_c = list(F_c) if isinstance(F_c, tuple) else F_c
        self.activation = get_activation(activation)

        with self.name_scope():
            self.embedding_atom = nn.Embedding(self.N_A, self.F_e)
            self.embedding_mask = nn.Embedding(3, self.F_e)
            self._build_graph_conv(*args, **kwargs)

            self.dense = nn.Sequential()
            for f_in, f_out in zip([self.F_skip] + self.F_c[:-1], self.F_c):
                self.dense.add(Linear_BN(f_in, f_out))

            self.policy_0 = self.params.get('policy_0', shape=[self.N_A, ], init=mx.init.Zero(), allow_deferred_init=False)
            self.policy_h = Policy(self.F_c[-1], self.Fh_policy, self.N_A, self.N_B)
        self.mode = 'loss'

    @abstractmethod
    def _build_graph_conv(self, *args, **kwargs): pass
    @abstractmethod
    def _graph_conv_forward(self, X, A): pass

    def _policy_0(self, ctx):
        logits = self.policy_0.data(ctx)
        return nd.softmax(logits, axis=0)

    def _policy(self, X, A, NX, NX_rep, last_append_mask):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A)
        X = self.dense(X)
        return self.policy_h(X, NX, NX_rep)

    def _likelihood(self, init, append, connect, end, action_0, actions, iw_ids, log_p_sigma, batch_size, iw_size, tox_class_batch):
        action_type, node_type, edge_type, append_pos, connect_pos = actions[:, 0], actions[:, 1], actions[:, 2], actions[:, 3], actions[:, 4]
        _log_mask = lambda _x, _mask: _mask * nd.log(_x + 1e-10) + (1 - _mask) * nd.zeros_like(_x)

        init = init.reshape([batch_size * iw_size, self.N_A])
        index = nd.stack(nd.arange(action_0.shape[0], ctx=action_0.context, dtype='int32'), action_0, axis=0)
        loss_init = nd.log(nd.gather_nd(init, index) + 1e-10)

        loss_end = _log_mask(end, nd.cast(action_type == 2, 'float32'))
        
        index = nd.stack(append_pos, node_type, edge_type, axis=0)
        loss_append = _log_mask(nd.gather_nd(append, index), nd.cast(action_type == 0, 'float32'))

        index = nd.stack(connect_pos, edge_type, axis=0)
        loss_connect = _log_mask(nd.gather_nd(connect, index), nd.cast(action_type == 1, 'float32'))

        log_p_x = loss_end + loss_append + loss_connect
        log_p_x = squeeze(SegmentSumFn(iw_ids, batch_size * iw_size)(unsqueeze(log_p_x, -1)), -1)
        log_p_x = log_p_x + loss_init

        log_p_x = log_p_x.reshape([batch_size, iw_size])
        log_p_sigma = log_p_sigma.reshape([batch_size, iw_size])
        
        # Original CDGCN log-likelihood calculation
        l = log_p_x - log_p_sigma
        l = logsumexp(l, axis=1) - math.log(float(iw_size))

        # =========================================================================
        # CTDDG Eq. 8 Implementation Fix
        # =========================================================================
        tox_labels = tox_class_batch.reshape((batch_size, iw_size))[:, 0]
        tox_weight = 1.0 - (1.0 + LAMBDA_TOX) * tox_labels
        l_weighted = l * tox_weight
        
        return l_weighted

    def forward(self, *input):
        if self.mode in ['loss', 'likelihood']:
            # BUG FIX: Corrected unpacking order so batch_size and iw_size map to the ints, and tox_class to the array
            X, A, iw_ids, last_append_mask, NX, NX_rep, action_0, actions, log_p, batch_size, iw_size, tox_class = input[:12]
            
            init = self._policy_0(X.context).tile([batch_size * iw_size, 1])
            append, connect, end = self._policy(X, A, NX, NX_rep, last_append_mask, *input[12:])
            
            l_weighted = self._likelihood(init, append, connect, end, action_0, actions, iw_ids, log_p, batch_size, iw_size, tox_class)
            return l_weighted if self.mode == 'likelihood' else -l_weighted.mean()


class MoleculeGenerator_RNN(MoleculeGenerator):
    __metaclass__ = ABCMeta
    def __init__(self, N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, *args, **kwargs):
        super(MoleculeGenerator_RNN, self).__init__(N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, *args, **kwargs)
        self.N_rnn = N_rnn
        with self.name_scope():
            self.rnn = gluon.rnn.GRU(hidden_size=self.F_c[-1], num_layers=self.N_rnn, layout='NTC', input_size=self.F_c[-1] * 2)

    def _rnn_train(self, X, NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum):
        X_avg = SegmentSumFn(NX_rep, NX.shape[0])(X) / nd.cast(unsqueeze(NX, 1), 'float32')
        X = nd.concat(X_avg, nd.take(X, indices=NX_cum - 1), dim=1)
        X = nd.take(X, indices=graph_to_rnn)
        batch_size, iw_size, length, num_features = X.shape
        
        X = self.rnn(X.reshape([batch_size * iw_size, length, num_features]))
        return nd.gather_nd(X.reshape([batch_size, iw_size, length, -1]), indices=rnn_to_graph)

    def _policy(self, X, A, NX, NX_rep, last_append_mask, graph_to_rnn, rnn_to_graph, NX_cum):
        X = self.embedding_atom(X) + self.embedding_mask(last_append_mask)
        X = self._graph_conv_forward(X, A)
        X_mol = self._rnn_train(self.dense(X), NX, NX_rep, graph_to_rnn, rnn_to_graph, NX_cum)
        return self.policy_h(self.dense(X), NX, NX_rep, X_mol)


class VanillaMolGen_RNN(MoleculeGenerator_RNN):
    def __init__(self, N_A, N_B, D, F_e, F_h, F_skip, F_c, Fh_policy, activation, N_rnn):
        super(VanillaMolGen_RNN, self).__init__(N_A, N_B, D, F_e, F_skip, F_c, Fh_policy, activation, N_rnn, F_h)

    def _build_graph_conv(self, F_h):
        self.F_h = list(F_h) if isinstance(F_h, tuple) else F_h
        self.conv, self.bn = [], []
        for i, (f_in, f_out) in enumerate(zip([self.F_e] + self.F_h[:-1], self.F_h)):
            conv = GraphConv(f_in, f_out, self.N_B + self.D)
            self.conv.append(conv)
            self.register_child(conv)
            bn = BatchNorm(in_channels=f_in) if i != 0 else None
            self.register_child(bn) if bn else None
            self.bn.append(bn)
            
        self.bn_skip = BatchNorm(in_channels=sum(self.F_h))
        self.linear_skip = Linear_BN(sum(self.F_h), self.F_skip)

    def _graph_conv_forward(self, X, A):
        X_out = [X]
        for conv, bn in zip(self.conv, self.bn):
            X = X_out[-1]
            X_out.append(conv(self.activation(bn(X)), A) if bn is not None else conv(X, A))
        X_out = nd.concat(*X_out[1:], dim=1)
        return self.activation(self.linear_skip(self.activation(self.bn_skip(X_out))))

## Initialization & Resume Logic

In [13]:
configs = {
    'F_e': F_E,           
    'F_h': F_H,           
    'F_skip': F_SKIP, 
    'F_c': F_C,
    'Fh_policy': FH_POLICY, 
    'activation': ACTIVATION, 
    'N_rnn': N_RNN
}

is_continuous = False
global_counter = 0

if not FORCE_RESTART and all(os.path.isfile(f) for f in [CKPT_PARAMS_FILE, CKPT_TRAINER_FILE, LOG_FILE]):
    is_continuous = True
    print("[INFO] Existing checkpoints found. Attempting to resume run.")
    with open(CKPT_CONFIG_FILE, 'r') as f:
        configs = json.load(f)
else:
    print("[INFO] Starting fresh run. Overwriting previous logs/checkpoints.")
    with open(CKPT_CONFIG_FILE, 'w') as f:
        json.dump(configs, f)

model = VanillaMolGen_RNN(
    get_mol_spec().num_atom_types, 
    get_mol_spec().num_bond_types, 
    D=D_RECEPTIVE, 
    **configs
)

if is_continuous:
    model.load_parameters(CKPT_PARAMS_FILE, ctx=CTX)
else:
    model.collect_params().initialize(mx.init.Xavier(), force_reinit=True, ctx=CTX)

opt = mx.optimizer.Adam(learning_rate=LR, clip_gradient=CLIP_GRAD)
trainer = gluon.Trainer(model.collect_params(), opt)

if is_continuous:
    trainer.load_states(CKPT_TRAINER_FILE)
    
    # ROBUST LOG PARSING: Handle empty or header-only log files
    with open(LOG_FILE, 'r') as f:
        records = [line.strip() for line in f.readlines() if line.strip()]
        
    if len(records) > 1: # We have actual data rows beyond the header
        final_record = records[-1] if records[-1] != 'Training finished' else records[-2]
        try:
            global_counter = int(final_record.split('\t')[0])
            print(f"[INFO] Successfully loaded states. Resuming from Iteration {global_counter}")
        except ValueError:
            print("[WARNING] Could not parse iteration from log. Starting from Iteration 0.")
            global_counter = 0
    else:
        print("[INFO] Log file only contains header. Resuming from Iteration 0.")
        global_counter = 0

num_params = sum(p.data().size for p in model.collect_params().values())
print(f"[INFO] Model initialized! Total trainable parameters: {num_params:,}")

[INFO] Starting fresh run. Overwriting previous logs/checkpoints.
[INFO] Model initialized! Total trainable parameters: 6,986,506


# **The Pretraining Loop**

## Pretraining Loop Setup

In [14]:
print("-" * 50)
print(f"Starting Pretraining on Context: {CTX}")
print(f"Batch Size: {BATCH_SIZE}, K Paths: {K_PATHS}, Alpha: {P_ALPHA}, Lambda: {LAMBDA_TOX}")
print("-" * 50)

# Prepare DataLoader Iterator
it_train = iter(loader_train)

# Setup Time Tracking
if not is_continuous:
    t0 = time.time()
    # Initialize Log File Header
    with open(LOG_FILE, 'w') as f:
        f.write('step\ttime(h)\tloss\tlr\n')
else:
    # Recover time offset from the last checkpoint
    with open(LOG_FILE, 'r') as f:
        records = [line.strip() for line in f.readlines() if line.strip()]
    
    if len(records) > 1:
        final_record = records[-1] if records[-1] != 'Training finished' else records[-2]
        try:
            # Extract the last recorded time (in hours) and adjust t0
            last_time_hours = float(final_record.split('\t')[1])
            t0 = time.time() - (last_time_hours * 3600)
            print(f"[INFO] Resuming timer from {last_time_hours:.2f} hours.")
        except ValueError:
            print("[WARNING] Could not parse time from log. Starting timer from 0.")
            t0 = time.time()
    else:
        print("Log file only contains header. Starting timer from 0.")
        t0 = time.time()

--------------------------------------------------
Starting Pretraining on Context: gpu(0)
Batch Size: 16, K Paths: 5, Alpha: 0.8, Lambda: 1.0
--------------------------------------------------


## Main Training Loop

In [ ]:
while global_counter < TOTAL_ITERATIONS:
    global_counter += 1
    
    # 1. Fetch next batch safely
    try:
        inputs_cpu = next(it_train)
    except StopIteration:
        it_train = iter(loader_train)
        inputs_cpu = next(it_train)

    # 2. Transfer to GPU
    inputs_gpu = batch_to_tensor(inputs_cpu)

    # 3. Forward and Backward Pass
    with autograd.record():
        # model() returns -mean(l_weighted) when mode='loss'
        loss = model(*inputs_gpu)
        loss.backward()


    # 4. Optimizer Step
    trainer.step(batch_size=1)
    
    # 5. Learning Rate Decay Step
    if global_counter % DECAY_STEP == 0:
        trainer.set_learning_rate(trainer.learning_rate * (1.0 - DECAY))

    # 6. Periodic Logging and Checkpointing
    if global_counter % SUMMARY_STEP == 0:
        loss_val = loss.asnumpy().item()
        elapsed_hours = float(time.time() - t0) / 3600.0
        
        # Save Model Parameters and Optimizer States
        model.save_parameters(CKPT_PARAMS_FILE)
        trainer.save_states(CKPT_TRAINER_FILE)

        # Write to log file
        log_str = f"{global_counter}\t{elapsed_hours:.4f}\t{loss_val:.4f}\t{trainer.learning_rate}\n"
        with open(LOG_FILE, 'a') as f:
            f.write(log_str)
            f.flush()

        print(f"Iter: {global_counter:06d} | Loss: {loss_val:.4f} | LR: {trainer.learning_rate:.6e} | Time: {elapsed_hours:.2f}h")

        # Strict cleanup for summary step
        del loss, inputs_cpu, inputs_gpu
        gc.collect()

## Final Save & Shutdown

In [ ]:
print("Target iterations reached. Saving final checkpoint...")
model.save_parameters(CKPT_PARAMS_FILE)
trainer.save_states(CKPT_TRAINER_FILE)

with open(LOG_FILE, 'a') as f:
    f.write('Training finished\n')

print("Pretraining Complete!")